# Word Embeddings : le modèle Word2Vec

## Imports

In [1]:
import sys

from gensim.models.phrases import Phrases, Phraser
from gensim.models import Word2Vec

import nltk
from nltk.tokenize import wordpunct_tokenize
from unidecode import unidecode

## Chargement et traitement des phrases du corpus

### Création d'un objet qui *streame* les lignes d'un fichier pour économiser de la RAM

In [2]:
class MySentences(object):
    """Tokenize and Lemmatize sentences"""
    def __init__(self, filename):
        self.filename = filename

    def __iter__(self):
        for line in open(self.filename, encoding='utf-8', errors="backslashreplace"):
            yield [unidecode(w.lower()) for w in wordpunct_tokenize(line)]

In [3]:
infile = "../../data/sents.txt"
sentences = MySentences(infile)

In [4]:
# Échantillon pour comparer les modèles : une phrase sur dix
class Echantillon(object):
    """Lit le fichier et garde une phrase sur `pas`, pour couvrir toutes les années."""
    def __init__(self, filename, pas=10):
        self.filename = filename
        self.pas = pas

    def __iter__(self):
        with open(self.filename, encoding='utf-8', errors="backslashreplace") as f:
            for i, line in enumerate(f):
                if i % self.pas == 0:
                    yield [unidecode(w.lower()) for w in wordpunct_tokenize(line)]

echantillon = list(Echantillon(infile, pas=10))
print(len(echantillon), "phrases dans l'échantillon")
print(sum(len(p) for p in echantillon), "mots dans l'échantillon")
print(echantillon[:3])

1205637 phrases dans l'échantillon
15365331 mots dans l'échantillon
[['mi', 'imnri', 'r', 'i', '<<', 'i', 'i', 'hmu', "'", 'i', '/', 'tx', "-'", 'l', ':', 'marche', 'tenu', 'hors', 'villa', ',', 'la', '9', '.'], ['da', '370', 'i', '6lutr', '.'], ['g', '.--;', 'paille', ',', 'v', '.', '--;', 'fcin', '.', '6', '.', '50', ':', 'lioumon', ',', '--.']]


### Détection des bigrams

Article intéressant sur le sujet : https://towardsdatascience.com/word2vec-for-phrases-learning-embeddings-for-more-than-one-word-727b6cf723cf

In [5]:
bigram_phrases = Phrases(echantillon)

L'object `phrases` peut être vu comme un large dictionnaire d'expressions multi-mots associées à un score, le *PMI-like scoring*. Ce dictionnaire est construit par un apprentissage sur base d'exemples.
Voir les références ci-dessous :
- https://arxiv.org/abs/1310.4546
- https://en.wikipedia.org/wiki/Pointwise_mutual_information

In [6]:
type(bigram_phrases.vocab)

dict

Il contient de nombreuses clés qui sont autant de termes observés dans le corpus

In [7]:
len(bigram_phrases.vocab.keys())

4297289

Prenons une clé au hasard :

In [8]:
key_ = list(bigram_phrases.vocab.keys())[144]
print(key_)

iloudmont_,


Le dictionnaire indique le score de cette coocurrence :

In [9]:
bigram_phrases.vocab[key_]

1

Lorsque l'instance de `Phrases` a été entraînée, elle peut concaténer les bigrams dans les phrases lorsque c'est pertinent.

### Conversion des `Phrases` en objet `Phraser`

`Phraser` est un alias pour `gensim.models.phrases.FrozenPhrases`, voir ici https://radimrehurek.com/gensim/models/phrases.html.

Le `Phraser` est une version *light* du `Phrases`, plus optimale pour transformer les phrases en concaténant les bigrams.

In [10]:
bigram_phraser = Phraser(phrases_model=bigram_phrases)

Le `Phraser` est un objet qui convertit certains unigrams d'une liste en bigrams lorsqu'ils ont été identifiés comme pertinents.

### Extraction des trigrams

Nous répétons l'opération en envoyant cette fois la liste de bigrams afin d'extraire les trigrams.

In [11]:
trigram_phrases = Phrases(bigram_phraser[echantillon])

In [12]:
trigram_phraser = Phraser(phrases_model=trigram_phrases)

### Création d'un corpus d'unigrams, bigrams, trigrams

In [13]:
corpus = list(trigram_phraser[bigram_phraser[echantillon]])

In [14]:
print(corpus[:10])

[['mi', 'imnri', 'r', 'i', '<<', 'i', 'i', 'hmu', "'", 'i', '/', 'tx', "-'", 'l', ':', 'marche', 'tenu', 'hors', 'villa', ',', 'la', '9', '.'], ['da', '370', 'i', '6lutr', '.'], ['g', '.--;_paille', ',', 'v', '.', '--;', 'fcin', '.', '6', '.', '50', ':', 'lioumon', ',', '--.'], ['17', '.--', 'a', '--;', 'farine', 'ira', 'de', 'froment', ',', '0', '.--', 'a', '0', '.'], ['2', '.--', 'a', '3', '.', '95', ';', 'poulet', ',', '2', '.', '25', 'a', '3', '.'], ['facteur', 'rural', '.'], ['ilendrickx', ',', 'von', 'glaboke', ',', 'sou', '-', "'", 'moy', '.', 'couturieaux', ',', 'lucion', ',', 'mersch', ',', 'iledo', ',', 'iloudmont', ',', 'de', 'heusch', '.'], ['enseignement_moyen', '.'], ['--', 'sont_nommes', 'membres', 'correspondants', 'de', 'la_commission', 'medicale', 'piminciale', 'de', 'namur', ';', 'mm', '.'], ['universite_populaire', 'd', "'", 'euerbeek', '.']]


In [15]:
from collections import Counter

# Compter les expressions de plusieurs mots détectées (elles contiennent un "_")
expressions = Counter(w for phrase in corpus for w in phrase if "_" in w)
print(len(expressions), "expressions différentes détectées")
print(expressions.most_common(30))

36449 expressions différentes détectées
[('1_/', 8643), ('000_fr', 7289), ('on_dem', 5972), ('a_louer', 5902), ('_', 5308), ('a_vendre', 4850), ('agence_rossel', 3314), ('il_y', 3271), ("jusqu_'", 3261), ('o_n_dem', 2773), ('000_f', 2633), ('on_demande', 2513), ('tous_les', 2426), ('tout_faire', 2294), ('00_;', 2232), ('se_pres', 2166), ('000_francs', 2147), ('/_2', 2039), ("aujourd_'", 2037), ('ce_qui', 2035), ('ont_ete', 2007), ('au_cours', 1797), ('demi_-', 1796), ('20_h', 1698), ('un_peu', 1636), ('o_n', 1575), ('la_ville', 1558), ('ce_que', 1557), ('le_soir', 1451), ('il_faut', 1439)]


## Entrainement d'un modèle Word2Vec sur ce corpus

In [16]:
%%time
model = Word2Vec(
    corpus, # On passe le corpus de ngrams que nous venons de créer
    vector_size=32, # Le nombre de dimensions dans lesquelles le contexte des mots devra être réduit, aka. vector_size
    window=5, # La taille du "contexte", ici 5 mots avant et après le mot observé
    min_count=5, # On ignore les mots qui n'apparaissent pas au moins 5 fois dans le corpus
    workers=4, # Permet de paralléliser l'entraînement du modèle en 4 threads
    epochs=5 # Nombre d'itérations du réseau de neurones sur le jeu de données pour ajuster les paramètres avec la descente de gradient, aka. epochs.
)

CPU times: total: 2min 2s
Wall time: 43.3 s


In [17]:
import numpy as np

# Paires de mots qui devraient être proches, et paires qui devraient être éloignées
# (les mots sont écrits sans accent et en minuscules, comme dans le corpus)
paires_proches = [("lundi", "mardi"), ("janvier", "fevrier"), ("bruxelles", "anvers"),
                  ("roi", "reine"), ("france", "allemagne"), ("ministre", "gouvernement"),
                  ("pere", "mere"), ("train", "gare")]
paires_eloignees = [("lundi", "farine"), ("roi", "kilo"), ("janvier", "cheval"),
                    ("ministre", "beurre")]

def evaluer(modele, nom):
    """Mesure la qualité d'un modèle avec les paires de test."""
    proches = [modele.wv.similarity(a, b) for a, b in paires_proches
               if a in modele.wv and b in modele.wv]
    eloignees = [modele.wv.similarity(a, b) for a, b in paires_eloignees
                 if a in modele.wv and b in modele.wv]
    resultat = {"modèle": nom,
                "vocabulaire": len(modele.wv),
                "proches": round(float(np.mean(proches)), 3),
                "éloignées": round(float(np.mean(eloignees)), 3)}
    resultat["écart"] = round(resultat["proches"] - resultat["éloignées"], 3)
    print(resultat)
    return resultat

resultats_w2v = [evaluer(model, "base : window=5, min_count=5")]

# Regarder aussi les voisins de trois mots
for mot in ["congo", "lundi", "ministre"]:
    print(mot, "->", model.wv.most_similar(mot, topn=5))

{'modèle': 'base : window=5, min_count=5', 'vocabulaire': 105013, 'proches': 0.711, 'éloignées': 0.018, 'écart': 0.693}
congo -> [('katanga', 0.8483808040618896), ('caire', 0.8269287347793579), ('cim', 0.8114081025123596), ('trust', 0.7972843647003174), ('congo_belge', 0.7861189246177673)]
lundi -> [('jeudi', 0.9886553883552551), ('vendredi', 0.9877122044563293), ('mercredi', 0.9843565225601196), ('mardi', 0.9836315512657166), ('samedi', 0.977409303188324)]
ministre -> [('president_du_conseil', 0.908436119556427), ('president', 0.9038641452789307), ('secretaire_general', 0.898812472820282), ('depute', 0.8797300457954407), ('conseiller', 0.876947820186615)]


In [18]:
# Comparer plusieurs réglages : un seul changement à la fois par rapport au modèle de base
reglages = [(2, 5), (10, 5), (5, 20), (5, 50)]
modeles_w2v = {(5, 5): model}

for window, min_count in reglages:
    m = Word2Vec(corpus, vector_size=32, window=window, min_count=min_count,
                 workers=4, epochs=5)
    modeles_w2v[(window, min_count)] = m
    resultats_w2v.append(evaluer(m, f"window={window}, min_count={min_count}"))
    print("   congo ->", [mot for mot, score in m.wv.most_similar("congo", topn=5)])

# Tableau récapitulatif des cinq modèles
import pandas as pd
pd.DataFrame(resultats_w2v)

{'modèle': 'window=2, min_count=5', 'vocabulaire': 105013, 'proches': 0.753, 'éloignées': 0.176, 'écart': 0.577}
   congo -> ['caire', 'katanga', 'bresil', 'cap', 'trust']
{'modèle': 'window=10, min_count=5', 'vocabulaire': 105013, 'proches': 0.712, 'éloignées': -0.064, 'écart': 0.776}
   congo -> ['katanga', 'caire', 'trust', 'congo_belge', 'charbonnages']
{'modèle': 'window=5, min_count=20', 'vocabulaire': 35766, 'proches': 0.709, 'éloignées': -0.011, 'écart': 0.72}
   congo -> ['katanga', 'caire', 'cim', 'kasai', 'cong']
{'modèle': 'window=5, min_count=50', 'vocabulaire': 16326, 'proches': 0.695, 'éloignées': -0.056, 'écart': 0.751}
   congo -> ['caire', 'katanga', 'trust', 'bresil', 'rupel']


,modèle,vocabulaire,proches,éloignées,écart
0,"base : window=5, min_count=5",105013,0.711,0.018,0.693
1,"window=2, min_count=5",105013,0.753,0.176,0.577
2,"window=10, min_count=5",105013,0.712,-0.064,0.776
3,"window=5, min_count=20",35766,0.709,-0.011,0.720
4,"window=5, min_count=50",16326,0.695,-0.056,0.751


In [19]:
# Itération 2 : combiner la grande fenêtre avec un seuil plus élevé
for window, min_count in [(10, 20), (10, 50)]:
    m = Word2Vec(corpus, vector_size=32, window=window, min_count=min_count,
                 workers=4, epochs=5)
    modeles_w2v[(window, min_count)] = m
    resultats_w2v.append(evaluer(m, f"window={window}, min_count={min_count}"))
    print("   congo ->", [mot for mot, score in m.wv.most_similar("congo", topn=5)])
    print("   ministre ->", [mot for mot, score in m.wv.most_similar("ministre", topn=5)])

pd.DataFrame(resultats_w2v)

{'modèle': 'window=10, min_count=20', 'vocabulaire': 35766, 'proches': 0.721, 'éloignées': -0.076, 'écart': 0.797}
   congo -> ['katanga', 'caire', 'congo_belge', 'trust', 'cong']
   ministre -> ['secretaire_general', 'la_presidence', 'rapporteur', 'depute', 'premier_ministre']
{'modèle': 'window=10, min_count=50', 'vocabulaire': 16326, 'proches': 0.742, 'éloignées': -0.091, 'écart': 0.833}
   congo -> ['katanga', 'caire', 'congo_belge', 'trust', 'colon']
   ministre -> ['president_du_conseil', 'premier_ministre', 'depute', 'secretaire_general', 'rapporteur']


,modèle,vocabulaire,proches,éloignées,écart
0,"base : window=5, min_count=5",105013,0.711,0.018,0.693
1,"window=2, min_count=5",105013,0.753,0.176,0.577
2,"window=10, min_count=5",105013,0.712,-0.064,0.776
3,"window=5, min_count=20",35766,0.709,-0.011,0.720
4,"window=5, min_count=50",16326,0.695,-0.056,0.751
5,"window=10, min_count=20",35766,0.721,-0.076,0.797
6,"window=10, min_count=50",16326,0.742,-0.091,0.833


In [22]:
# Les mots que nous voulons explorer sont-ils dans le vocabulaire des deux candidats ?
mots_test = ["lumumba", "tshombe", "kasavubu", "mobutu", "baudouin", "fabiola",
             "kennedy", "khrouchtchev", "eyskens", "spaak", "hitler", "leopold",
             "greve", "independance", "colonie", "femme", "ouvrier", "paix"]

for reglage in [(10, 20), (10, 50)]:
    m = modeles_w2v[reglage]
    absents = [w for w in mots_test if w not in m.wv]
    print(f"window={reglage[0]}, min_count={reglage[1]} : "
          f"{len(mots_test) - len(absents)} mots présents sur {len(mots_test)}")
    print("   absents :", absents)

# Nombre d'apparitions de chaque mot dans l'échantillon
m20 = modeles_w2v[(10, 20)]
print({w: m20.wv.get_vecattr(w, "count") for w in mots_test if w in m20.wv})

window=10, min_count=20 : 13 mots présents sur 18
   absents : ['lumumba', 'tshombe', 'kasavubu', 'mobutu', 'fabiola']
window=10, min_count=50 : 11 mots présents sur 18
   absents : ['lumumba', 'tshombe', 'kasavubu', 'mobutu', 'fabiola', 'kennedy', 'eyskens']
{'baudouin': np.int64(202), 'kennedy': np.int64(46), 'khrouchtchev': np.int64(59), 'eyskens': np.int64(32), 'spaak': np.int64(119), 'hitler': np.int64(170), 'leopold': np.int64(927), 'greve': np.int64(131), 'independance': np.int64(231), 'colonie': np.int64(91), 'femme': np.int64(2197), 'ouvrier': np.int64(1460), 'paix': np.int64(870)}


In [23]:
%%time
# Appliquer la détection des expressions à tout le corpus, puis l'écrire dans un fichier.
# Les phrases sont lues et écrites une par une, sans tout charger en mémoire.
corpus_complet = "../../data/sents_ngrams.txt"
n = 0
with open(corpus_complet, "w", encoding="utf-8") as sortie:
    for phrase in trigram_phraser[bigram_phraser[sentences]]:
        sortie.write(" ".join(phrase) + "\n")
        n += 1
print(n, "phrases écrites dans", corpus_complet)

12056370 phrases écrites dans ../../data/sents_ngrams.txt
CPU times: total: 8min 10s
Wall time: 8min 13s


In [24]:
%%time
# Entraîner le modèle final sur tout le corpus, avec le réglage retenu
model_final = Word2Vec(corpus_file=corpus_complet, vector_size=32, window=10,
                       min_count=200, workers=8, epochs=5)

evaluer(model_final, "final : corpus complet, window=10, min_count=200")

# Enregistrer le modèle pour l'exploration
model_final.save("../../data/newspapers.model")

{'modèle': 'final : corpus complet, window=10, min_count=200', 'vocabulaire': 34820, 'proches': 0.735, 'éloignées': -0.054, 'écart': 0.789}
CPU times: total: 47min 23s
Wall time: 7min


#### Remarque

Vous voyez ici que l'entrainement du modèle est parallélisé (sur 4 workers).

Lors qu'on parallélise l'entrainement du modèle, 4 modèles "séparés" sont entrainés sur environ un quart des phrases.

Ensuite, les résultats sont agrégés pour ne plus faire qu'un seul modèle.

On ne peut prédire quel worker aura quelle phrase, car il y a des aléas lors de la parallélisation (p. ex. un worker qui serait plus lent, etc.).

Du coup, les valeurs peuvent varier légèrement d'un entrainement à l'autre.

Mais, globalement, les résultats restent cohérents.

### Sauver le modèle dans un fichier

## Explorer le modèle

### Charger le modèle en mémoire

In [25]:
model = Word2Vec.load("../../data/newspapers.model")

In [26]:
# 1. Similarité entre deux mots (de -1 à 1)
paires = [("roi", "reine"), ("belgique", "congo"), ("guerre", "paix"),
          ("ouvrier", "patron"), ("homme", "femme"), ("bruxelles", "paris"),
          ("greve", "syndicat")]
print("=== similarity ===")
for a, b in paires:
    if a in model.wv and b in model.wv:
        print(f"{a} - {b} : {model.wv.similarity(a, b):.3f}")
    else:
        print(f"{a} - {b} : un des deux mots est absent du vocabulaire")

# 2. Les mots les plus proches d'un mot donné
print("\n=== most_similar ===")
for mot in ["congo", "guerre", "femme", "greve", "independance"]:
    if mot in model.wv:
        voisins = [(w, round(float(s), 2)) for w, s in model.wv.most_similar(mot, topn=8)]
        print(mot, "->", voisins)
    else:
        print(mot, "-> absent du vocabulaire")

# 3. Une analogie : roi - homme + femme = ?
print("\n=== analogie ===")
print(model.wv.most_similar(positive=["roi", "femme"], negative=["homme"], topn=3))

=== similarity ===
roi - reine : 0.605
belgique - congo : 0.407
guerre - paix : 0.428
ouvrier - patron : 0.580
homme - femme : 0.612
bruxelles - paris : 0.457
greve - syndicat : 0.383

=== most_similar ===
congo -> [('congo_belge', 0.83), ('colon', 0.82), ('katanga', 0.82), ('klvu', 0.82), ('belg', 0.8), ('trust', 0.79), ('kivu', 0.79), ('banq', 0.79)]
guerre -> [('guerro', 0.82), ('la_liberation', 0.81), ('armee', 0.79), ('la_guerre', 0.79), ('liberation', 0.79), ('pendant_la_guerre', 0.78), ('armistice', 0.78), ('concentration', 0.77)]
femme -> [('femmo', 0.94), ('jeune_fille', 0.91), ('jeune_femme', 0.87), ('bonne_femme', 0.85), ('nourrice', 0.85), ('fille', 0.85), ('mariee', 0.85), ('gouvernante', 0.81)]
greve -> [('la_greve', 0.87), ('une_greve', 0.8), ('les_mineurs', 0.8), ('les_greves', 0.78), ('des_mineurs', 0.76), ('la_continuation', 0.76), ('en_prevision', 0.75), ('les_operations', 0.73)]
independance -> [('alliance', 0.87), ('unite', 0.85), ('abdication', 0.85), ('annexion',

### Imprimer le vecteur d'un terme

In [27]:
model.wv["ministre"]

array([-0.01542065,  3.0529094 ,  5.36967   ,  2.4634233 , -2.42082   ,
       -7.7732444 ,  9.864565  , -0.56964606,  6.38455   , -4.568495  ,
        9.015496  ,  0.9474776 ,  2.3935068 , -5.138426  ,  1.9492965 ,
       -2.3201296 , -0.52516085,  3.2250886 , -5.192951  , -3.5664806 ,
       -3.7146049 ,  4.868607  , -3.162234  , -4.862856  , -1.2776227 ,
        0.9289848 ,  0.4392335 , -9.439494  ,  3.4460976 , -6.511631  ,
       -0.96880686,  2.2713265 ], dtype=float32)

### Calculer la similarité entre deux termes

In [28]:
model.wv.similarity("ministre", "roi")

np.float32(0.50755465)

### Chercher les mots les plus proches d'un terme donné

In [29]:
model.wv.most_similar("ministre", topn=10)

[('president_du_conseil', 0.9176390767097473),
 ('premier_ministre', 0.90163254737854),
 ('ministre_des_finances', 0.8911773562431335),
 ('depute', 0.8720540404319763),
 ('president', 0.8602399230003357),
 ('nouveau_ministre', 0.8575522303581238),
 ('secretaire_general', 0.854069709777832),
 ('conseiller', 0.8531702756881714),
 ('le_ministre', 0.8491538763046265),
 ('du_ministre', 0.8463058471679688)]

### Faire des recherches complexes à travers l'espace vectoriel

In [31]:
# Analogies de capitales : on vérifie d'abord que les mots existent
analogies = [(["paris", "allemagne"], ["france"]),    # Paris - France + Allemagne = Berlin ?
             (["bruxelles", "france"], ["belgique"])] # Bruxelles - Belgique + France = Paris ?
for positifs, negatifs in analogies:
    mots = positifs + negatifs
    absents = [m for m in mots if m not in model.wv]
    if absents:
        print(positifs, "-", negatifs, ": mots absents", absents)
    else:
        print(positifs, "-", negatifs, "->", model.wv.most_similar(positive=positifs, negative=negatifs, topn=3))

['paris', 'allemagne'] - ['france'] -> [('alger', 0.7751353979110718), ('agence_reuter', 0.7627139687538147), ('ankara', 0.7610573172569275)]
['bruxelles', 'france'] - ['belgique'] -> [('du_jardin_botanique', 0.844542920589447), ('bischoffsheim', 0.8280832171440125), ('galilee', 0.7404676079750061)]
